In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from tqdm.auto import tqdm

In [11]:
RAW_PATH = "/home/shayan/Distributional-Hate-Speech-Prediction/data/raw/mhs_raw.csv"  
PROCESSED_PATH = "../data/processed/mhs_main_experiment_annotations_with_split.csv"

raw = pd.read_csv(RAW_PATH)
ann = pd.read_csv(PROCESSED_PATH)

print("Raw shape:", raw.shape)
print("Processed shape:", ann.shape)

Raw shape: (135556, 143)
Processed shape: (49433, 53)


In [12]:
[col for col in raw.columns if "hate" in col.lower() or "score" in col.lower()]

['hatespeech', 'hate_speech_score']

In [13]:
required_raw_cols = [
    "comment_id",
    "annotator_id",
    "hate_speech_score"
]

missing_raw_cols = [col for col in required_raw_cols if col not in raw.columns]
missing_raw_cols

[]

In [14]:
raw["hate_speech_score"].describe()

count    135556.000000
mean         -0.567428
std           2.380003
min          -8.340000
25%          -2.330000
50%          -0.340000
75%           1.410000
max           6.300000
Name: hate_speech_score, dtype: float64

In [15]:
raw.groupby("comment_id")["hate_speech_score"].describe().head()

,count,mean,std,min,25%,50%,75%,max
comment_id,,,,,,,,
1,4.0,0.46,0.0,0.46,0.46,0.46,0.46,0.46
2,3.0,0.03,0.0,0.03,0.03,0.03,0.03,0.03
3,3.0,-1.29,0.0,-1.29,-1.29,-1.29,-1.29,-1.29
4,2.0,-0.24,0.0,-0.24,-0.24,-0.24,-0.24,-0.24
5,3.0,-2.84,0.0,-2.84,-2.84,-2.84,-2.84,-2.84


In [16]:
score_variation = (
    raw.groupby("comment_id")["hate_speech_score"]
    .nunique()
    .reset_index(name="unique_scores_per_comment")
)

score_variation["unique_scores_per_comment"].value_counts().sort_index()

unique_scores_per_comment
1    39565
Name: count, dtype: int64

In [17]:
varying_score_comments = score_variation[
    score_variation["unique_scores_per_comment"] > 1
]["comment_id"].head(10)

raw[raw["comment_id"].isin(varying_score_comments)][
    ["comment_id", "annotator_id", "hate_speech_score"]
].sort_values("comment_id")

,comment_id,annotator_id,hate_speech_score


In [18]:
same_score_examples = score_variation[
    score_variation["unique_scores_per_comment"] == 1
]["comment_id"].head(10)

raw[raw["comment_id"].isin(same_score_examples)][
    ["comment_id", "annotator_id", "hate_speech_score"]
].sort_values("comment_id")

,comment_id,annotator_id,hate_speech_score
1064,1,3330,0.46
70119,1,712,0.46
50410,1,8185,0.46
23298,1,962,0.46
36676,2,4328,0.03
59653,2,5156,0.03
10872,2,8015,0.03
47847,3,4253,-1.29
25715,3,3856,-1.29
165,3,5021,-1.29


In [19]:
score_per_comment = (
    raw.groupby("comment_id")
    .agg(
        hate_speech_score=("hate_speech_score", "first"),
        unique_scores_per_comment=("hate_speech_score", "nunique")
    )
    .reset_index()
)

ann_score = ann.merge(
    score_per_comment,
    on="comment_id",
    how="left"
)

print("Merged shape:", ann_score.shape)
print("Missing hate_speech_score after merge:", ann_score["hate_speech_score"].isna().sum())

Merged shape: (49433, 55)
Missing hate_speech_score after merge: 0


In [20]:
comment_score_df = (
    ann_score.groupby("comment_id")
    .agg(
        text_clean=("text_clean", "first"),
        split=("split", "first"),
        target_type=("target_type", "first"),
        is_women_targeted=("is_women_targeted", "max"),
        is_immigrant_targeted=("is_immigrant_targeted", "max"),
        annotation_count=("annotator_id", "count"),
        unique_annotators=("annotator_id", "nunique"),
        hate_speech_score=("hate_speech_score", "first"),
        unique_scores_per_comment=("unique_scores_per_comment", "first")
    )
    .reset_index()
)

comment_score_df.head()

,comment_id,text_clean,split,target_type,is_women_targeted,is_immigrant_targeted,annotation_count,unique_annotators,hate_speech_score,unique_scores_per_comment
0,6,First off you look cool as fuck! Anyway if we ...,train,women_only,1,0,2,2,1.72,1
1,7,\*points to posters asking for palestinian rig...,test,immigrant_only,0,1,2,2,-0.77,1
2,10,"They'll come back in your plan, also. Plus we ...",train,immigrant_only,0,1,3,3,1.35,1
3,11,"eat my fuck, bitch",validation,women_only,1,0,2,2,1.07,1
4,12,She ugly anyways,train,women_only,1,0,2,2,0.76,1


In [21]:
score_per_comment = (
    raw.groupby("comment_id")
    .agg(
        hate_speech_score=("hate_speech_score", "first"),
        unique_scores_per_comment=("hate_speech_score", "nunique")
    )
    .reset_index()
)

ann_score = ann.merge(
    score_per_comment,
    on="comment_id",
    how="left"
)

print("Merged shape:", ann_score.shape)
print("Missing hate_speech_score after merge:", ann_score["hate_speech_score"].isna().sum())

Merged shape: (49433, 55)
Missing hate_speech_score after merge: 0


In [22]:
score_by_split = (
    comment_score_df
    .groupby("split")["hate_speech_score"]
    .describe()
)

score_by_split

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,2965.0,-0.722327,2.053684,-8.16,-1.97,-0.44,0.74,6.30
train,13832.0,-0.767610,2.043660,-8.30,-2.02,-0.48,0.71,6.05
validation,2964.0,-0.743009,1.986734,-8.14,-2.00,-0.48,0.68,5.10


In [23]:
score_by_target_type = (
    comment_score_df
    .groupby("target_type")["hate_speech_score"]
    .describe()
)

score_by_target_type

,count,mean,std,min,25%,50%,75%,max
target_type,,,,,,,,
immigrant_only,8003.0,-0.942784,2.027277,-8.30,-2.26,-0.740,0.5100,6.30
women_and_immigrant,384.0,-1.041302,2.415827,-7.93,-2.54,-0.595,0.7375,4.50
women_only,11374.0,-0.616898,2.018009,-8.28,-1.78,-0.280,0.8100,6.05


In [24]:
comment_score_df[["hate_speech_score", "annotation_count"]].corr()

,hate_speech_score,annotation_count
hate_speech_score,1.0000,0.0262
annotation_count,0.0262,1.0000


In [25]:
comment_score_df.groupby("annotation_count")["hate_speech_score"].describe().head(10)

,count,mean,std,min,25%,50%,75%,max
annotation_count,,,,,,,,
1,9122.0,-0.914233,2.092068,-8.30,-2.300,-0.695,0.5875,6.30
2,5995.0,-0.679867,1.948809,-8.07,-1.875,-0.420,0.7100,5.58
3,3409.0,-0.567750,1.985897,-7.94,-1.620,-0.230,0.8500,4.31
4,1148.0,-0.525540,2.043462,-8.28,-1.390,-0.105,0.8625,3.97
5,30.0,-0.405000,2.264263,-7.07,-0.985,0.085,0.9625,2.54
6,1.0,-3.980000,NaN,-3.98,-3.980,-3.980,-3.9800,-3.98
9,1.0,-3.010000,NaN,-3.01,-3.010,-3.010,-3.0100,-3.01
10,1.0,-5.400000,NaN,-5.40,-5.400,-5.400,-5.4000,-5.40
11,1.0,-1.500000,NaN,-1.50,-1.500,-1.500,-1.5000,-1.50


In [26]:
def make_distribution(group, label_col, label_values):
    counts = group[label_col].value_counts(normalize=True)
    return np.array([counts.get(label, 0.0) for label in label_values])

def entropy(probs):
    probs = probs[probs > 0]
    if len(probs) == 0:
        return 0.0
    return -np.sum(probs * np.log2(probs))

entropy_rows = []

for comment_id, group in ann_score.groupby("comment_id"):
    dist = make_distribution(group, "hatespeech", [0, 1, 2])
    entropy_value = entropy(dist)

    entropy_rows.append({
        "comment_id": comment_id,
        "hatespeech_entropy": entropy_value,
        "hatespeech_majority": group["hatespeech"].value_counts().idxmax()
    })

entropy_df = pd.DataFrame(entropy_rows)

comment_score_df = comment_score_df.merge(
    entropy_df,
    on="comment_id",
    how="left"
)

comment_score_df[[
    "hate_speech_score",
    "hatespeech_entropy",
    "annotation_count"
]].corr()

,hate_speech_score,hatespeech_entropy,annotation_count
hate_speech_score,1.000000,0.279131,0.026200
hatespeech_entropy,0.279131,1.000000,0.069257
annotation_count,0.026200,0.069257,1.000000


In [27]:
dist_rows = []

for comment_id, group in ann_score.groupby("comment_id"):
    dist = make_distribution(group, "hatespeech", [0, 1, 2])

    dist_rows.append({
        "comment_id": comment_id,
        "hatespeech_0_prob": dist[0],
        "hatespeech_1_prob": dist[1],
        "hatespeech_2_prob": dist[2],
    })

dist_df = pd.DataFrame(dist_rows)

comment_score_df = comment_score_df.merge(
    dist_df,
    on="comment_id",
    how="left"
)

comment_score_df[[
    "hate_speech_score",
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob",
    "hatespeech_entropy"
]].corr()

,hate_speech_score,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,hatespeech_entropy
hate_speech_score,1.000000,-0.554498,0.107899,0.538002,0.279131
hatespeech_0_prob,-0.554498,1.000000,-0.415892,-0.837142,-0.267775
hatespeech_1_prob,0.107899,-0.415892,1.000000,-0.149276,0.227626
hatespeech_2_prob,0.538002,-0.837142,-0.149276,1.000000,0.154238
hatespeech_entropy,0.279131,-0.267775,0.227626,0.154238,1.000000


In [28]:
score_by_majority = (
    comment_score_df
    .groupby("hatespeech_majority")["hate_speech_score"]
    .describe()
)

score_by_majority

,count,mean,std,min,25%,50%,75%,max
hatespeech_majority,,,,,,,,
0.0,13332.0,-1.462057,1.936646,-8.30,-2.71,-1.20,-0.0600,4.93
1.0,1462.0,-0.155520,1.386068,-6.85,-0.98,-0.04,0.7675,3.66
2.0,4967.0,0.957912,1.220531,-4.82,0.22,0.99,1.7100,6.30


In [29]:
comment_score_df["entropy_group"] = np.where(
    comment_score_df["hatespeech_entropy"] == 0,
    "zero_entropy",
    "nonzero_entropy"
)

score_by_entropy_group = (
    comment_score_df
    .groupby("entropy_group")["hate_speech_score"]
    .describe()
)

score_by_entropy_group

,count,mean,std,min,25%,50%,75%,max
entropy_group,,,,,,,,
nonzero_entropy,4455.0,0.308337,1.245747,-6.02,-0.42,0.41,1.13,4.18
zero_entropy,15306.0,-1.067242,2.115983,-8.30,-2.47,-0.87,0.45,6.30


In [30]:
score_reliable_subset = comment_score_df[
    comment_score_df["annotation_count"] >= 3
].copy()

print("Reliable subset comments:", len(score_reliable_subset))
print("Percent kept:", round(len(score_reliable_subset) / len(comment_score_df) * 100, 2))

score_reliable_subset["hate_speech_score"].describe()

Reliable subset comments: 4644
Percent kept: 23.5


count    4644.000000
mean       -0.548262
std         2.013287
min        -8.280000
25%        -1.570000
50%        -0.200000
75%         0.872500
max         4.310000
Name: hate_speech_score, dtype: float64

In [31]:
score_reliable_subset.groupby("split")["hate_speech_score"].describe()

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,705.0,-0.546312,2.113502,-8.16,-1.5500,-0.240,0.950,4.31
train,3252.0,-0.540864,2.006481,-8.28,-1.5425,-0.175,0.860,4.05
validation,687.0,-0.585284,1.941329,-7.73,-1.7400,-0.260,0.875,3.83


In [32]:
os.makedirs("../data/processed", exist_ok=True)

ann_score_path = "../data/processed/mhs_main_experiment_annotations_with_split_and_score.csv"
comment_score_path = "../data/processed/mhs_comment_level_with_hate_speech_score.csv"

ann_score.to_csv(ann_score_path, index=False)
comment_score_df.to_csv(comment_score_path, index=False)

print("Saved annotation-level with score:", ann_score_path)
print("Saved comment-level with score:", comment_score_path)

Saved annotation-level with score: ../data/processed/mhs_main_experiment_annotations_with_split_and_score.csv
Saved comment-level with score: ../data/processed/mhs_comment_level_with_hate_speech_score.csv


In [33]:
os.makedirs("../results/tables", exist_ok=True)

report_path = "../results/tables/hate_speech_score_audit.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("HATE_SPEECH_SCORE AUDIT\n")
    f.write("=" * 90 + "\n\n")

    f.write("1. PURPOSE\n")
    f.write("-" * 90 + "\n")
    f.write("This audit checks whether hate_speech_score is an annotator-level target or a repeated comment-level aggregate score.\n\n")

    f.write("2. RAW DATA SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(f"Raw rows: {len(raw)}\n")
    f.write(f"Raw unique comments: {raw['comment_id'].nunique()}\n")
    f.write(f"Raw unique annotators: {raw['annotator_id'].nunique()}\n\n")

    f.write("3. HATE_SPEECH_SCORE SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(raw["hate_speech_score"].describe().to_string())
    f.write("\n\n")

    f.write("4. SCORE VARIATION WITHIN COMMENT\n")
    f.write("-" * 90 + "\n")
    f.write(score_variation["unique_scores_per_comment"].value_counts().sort_index().to_string())
    f.write("\n\n")

    f.write("5. SCORE BY SPLIT\n")
    f.write("-" * 90 + "\n")
    f.write(score_by_split.to_string())
    f.write("\n\n")

    f.write("6. SCORE BY TARGET TYPE\n")
    f.write("-" * 90 + "\n")
    f.write(score_by_target_type.to_string())
    f.write("\n\n")

    f.write("7. CORRELATION WITH HATESPEECH DISTRIBUTION AND ENTROPY\n")
    f.write("-" * 90 + "\n")
    f.write(comment_score_df[[
        "hate_speech_score",
        "hatespeech_0_prob",
        "hatespeech_1_prob",
        "hatespeech_2_prob",
        "hatespeech_entropy",
        "annotation_count"
    ]].corr().to_string())
    f.write("\n\n")

    f.write("8. SCORE BY MAJORITY HATESPEECH LABEL\n")
    f.write("-" * 90 + "\n")
    f.write(score_by_majority.to_string())
    f.write("\n\n")

    f.write("9. SCORE BY ENTROPY GROUP\n")
    f.write("-" * 90 + "\n")
    f.write(score_by_entropy_group.to_string())
    f.write("\n\n")

    f.write("10. RELIABLE SUBSET SCORE SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(f"Reliable subset comments: {len(score_reliable_subset)}\n")
    f.write(f"Percent kept: {round(len(score_reliable_subset) / len(comment_score_df) * 100, 2)}%\n\n")
    f.write(score_reliable_subset["hate_speech_score"].describe().to_string())
    f.write("\n\n")

    f.write("11. INTERPRETATION\n")
    f.write("-" * 90 + "\n")

    if score_variation["unique_scores_per_comment"].max() == 1:
        f.write("hate_speech_score is constant within each comment. This means it is a repeated comment-level aggregate score, not an annotator-level target.\n")
        f.write("It is therefore suitable for regression/severity prediction, but it does not preserve annotator-level disagreement by itself.\n")
    else:
        f.write("Some comments contain multiple hate_speech_score values. This suggests the score may contain annotation-level variation and needs further inspection.\n")

    f.write("The score can be used as a stable severity target, while entropy/distributional labels should be used separately if the goal is to model disagreement.\n")

print("Saved:", report_path)
print(open(report_path, encoding="utf-8").read())

Saved: ../results/tables/hate_speech_score_audit.txt
HATE_SPEECH_SCORE AUDIT

1. PURPOSE
------------------------------------------------------------------------------------------
This audit checks whether hate_speech_score is an annotator-level target or a repeated comment-level aggregate score.

2. RAW DATA SUMMARY
------------------------------------------------------------------------------------------
Raw rows: 135556
Raw unique comments: 39565
Raw unique annotators: 7912

3. HATE_SPEECH_SCORE SUMMARY
------------------------------------------------------------------------------------------
count    135556.000000
mean         -0.567428
std           2.380003
min          -8.340000
25%          -2.330000
50%          -0.340000
75%           1.410000
max           6.300000

4. SCORE VARIATION WITHIN COMMENT
------------------------------------------------------------------------------------------
unique_scores_per_comment
1    39565

5. SCORE BY SPLIT
-------------------------------